# Navigating Information Overload in Healthcare

Healthcare professionals today face an overwhelming influx of patient data, ranging from Electronic Health Records (EHRs) and clinical notes to lab results and imaging reports. Processing this vast amount of information efficiently is often challenging, especially in critical situations.

### Illustrative Scenario

Consider a doctor reviewing a patient's EHR to inform a treatment decision. The EHR might encompass:

- 🧾 Several pages of discharge summaries  
- 💬 Progress notes from various specialists  
- 🧪 Weeks of lab results  
- 📋 Comprehensive medication history  
- 🖼️ Detailed radiology reports  

Below is a real-world excerpt from a progress note:

> *The patient has been experiencing arm pain for several months. An initial exercise stress echocardiogram was equivocal, followed by a nuclear stress test indicating inferobasilar ischemia. Although a heart catheterization was initially advised, the patient opted for medical therapy, beginning beta-blockers. However, arm pain symptoms worsened, and new jaw pain emerged, leading to an emergency room visit. On 08/16/08, a cardiac catheterization revealed normal left main, 80% mid LAD lesion, normal circumflex, and a totally occluded RCA in the mid-portion with collaterals from the left to the right. A transfer was arranged due to potential collateral insufficiency from LAD stenosis to the RCA. The patient underwent placement of three or four drug-eluting stents on 08/16/08 and was discharged on 08/17/08. Mild left arm discomfort occurred on 08/18/08. The following day, severe fatigue was noted without arm pain, but today, arm pain recurred after walking, resolving completely after three sublingual nitroglycerin doses. This is consistent with her usual angina. She is being admitted for unstable angina post-stent.*

Even this brief note contains crucial information embedded within much less critical data, highlighting the significant cognitive burden. This burden is amplified by dozens of similar notes per patient visit.

**This is precisely where summarization tools prove invaluable, enabling rapid and accurate distillation of essential clinical information.**

Here’s a potential summary:

> *The patient, presenting with chronic arm pain, initially chose medical management over heart catheterization after mixed stress test results. Worsening symptoms, including jaw pain, led to an ER visit. Cardiac catheterization on 08/16/08 revealed LAD stenosis and RCA occlusion, necessitating drug-eluting stent placement. Post-discharge, the patient experienced mild arm discomfort and fatigue, followed by exertion-induced arm pain (typical angina) requiring nitroglycerin. She is now admitted with unstable angina post-stenting.*

# Summarization Approaches:

Automatic text summarization typically employs two main strategies:

- Extractive Summarization
- Abstractive Summarization

### 🔍 Core Distinctions

| Feature                  | Extractive                                | Abstractive                                 |
|:-------------------------|:------------------------------------------|:--------------------------------------------|
| Output Style             | Direct copies of original text            | Generates entirely new textual content      |
| Coherence                | Can sometimes be fragmented               | Generally more fluid and natural            |
| Factual Reliability      | High (original sentences preserved)       | Moderate (potential for factual inaccuracies)|
| Interpretability         | Easy to trace back to source              | More challenging to trace                   |
| Computational Demand     | Lower                                     | Higher                                      |
| Domain Adaptability      | Moderate                                  | High (especially with fine-tuning)          |

---

➡️ Within healthcare, **extractive methods** are beneficial for quickly highlighting key information, while **abstractive summarization** can provide deeper insights and improved readability, particularly for clinical notes, discharge summaries, and radiology reports.

<h1 style="color:red;">Initial Setup</h1>

## Enabling the Accelerator
Platforms like Kaggle and Google Colab offer access to GPUs (accelerators), which significantly speed up computations for most resource-intensive models. To utilize this feature, you must enable the accelerator by following these steps:

1.  Click the "Runtime" menu on the top bar (located just below the notebook title).
2.  Select "Change runtime type."
3.  Choose "T4 GPU" from the options.
4.  Confirm your selection.

# First Steps: Loading a dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/singla007/MTSamples/refs/heads/main/mtsamples.csv',index_col=None)
df.drop_duplicates(subset=['description'],inplace=True)
df.head()

,description,medical_specialty,sample_name,transcription,keywords
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."
5,Morbid obesity. Laparoscopic antecolic anteg...,Bariatrics,Laparoscopic Gastric Bypass,"PREOPERATIVE DIAGNOSIS: , Morbid obesity.,POST...","bariatrics, gastric bypass, eea anastomosis, r..."


In [ ]:
print(df.shape)

(2348, 5)


# Extractive Summarization

**Definition:**  
Extractive summarization involves directly selecting and combining the most crucial sentences or phrases from the original document, without altering their wording.

**Strengths:**
-   Efficient and fast computational process.
-   Maintains factual accuracy by using original text segments.
-   Valuable when transparency and traceability are essential.

**Weaknesses:**
-   May lead to some redundancy or lack of overall cohesion.
-   Incapable of paraphrasing or synthesizing concepts.
-   Less effective at reducing verbosity compared to abstractive methods.

---

## Method 1: Extractive Summary using TF-IDF

### 🧠 Understanding TF-IDF for Extractive Summarization

Extractive summarization leveraging **TF-IDF (Term Frequency-Inverse Document Frequency)** is founded on a straightforward principle:

> 🗣️ **"Sentences containing words that are rare within the document but significant within that sentence itself are strong candidates for a summary."**

### ✍️ Operational Flow

1.  **Segment the text into individual sentences.**
2.  **Generate TF-IDF vectors for each sentence and the complete document**: Words that appear frequently in a specific sentence but infrequently across the entire set of sentences receive higher scores.
3.  **Calculate the cosine similarity** between each sentence's vector and the overall document's vector: This metric quantifies how representative a sentence is of the full text.
4.  **Identify the top N most similar sentences** to form the summary.

---

### 🖼️ Conceptual Visualization

Imagine a diagram where:
-   The full document is represented at the center.
-   Individual sentences are scattered around it.
-   Arrows connect each sentence to the document, with the arrow's thickness indicating the TF-IDF cosine similarity score.
-   The sentences most strongly connected (highest similarity) are highlighted as they are chosen for the summary.

*This illustration conceptually depicts sentences being ranked by their semantic alignment with the overall document. The most pertinent sentences are then chosen.*

---

In clinical narratives or medical records:
-   Terms like _"ischemia"_, _"occlusion"_, or _"stent"_ carry substantial clinical weight.
-   TF-IDF helps **identify these highly informative sentences** without the need for complex deep learning models.
-   This method is **quick**, **transparent**, and **easily interpretable**.

In [ ]:
idx = 0
text = df.iloc[idx]['transcription']
print(text)

SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal sprays.  She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate.  Nasal mucosa was erythematous and swollen.  Only clear drainage was seen.  TMs were clear.,Neck:  Supple without adenopathy.,L

In [ ]:
import nltk
# Download required resources
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import numpy as np
from pprint import pprint
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import heapq

# Step 1: Sentence Tokenization
sentences = nltk.sent_tokenize(text)

# Step 2: TF-IDF Vectorization
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)

# Step 3: Compute cosine similarity of each sentence with the whole text
document_vector = vectorizer.transform([text])
similarity_scores = cosine_similarity(tfidf_matrix, document_vector).flatten()

# Step 4: Select top-N most similar sentences
top_n = 1
top_indices = heapq.nlargest(top_n, range(len(similarity_scores)), similarity_scores.__getitem__)
summary = [sentences[i] for i in sorted(top_indices)]  # sort to maintain original order

# Display extractive summary
print("📌 Extractive Summary:\n")
for sent in summary:
    pprint(sent.strip())

📌 Extractive Summary:

('She does have asthma but doest not require daily medication for this and '
 'does not think it is flaring up.,MEDICATIONS: , Her only medication '
 'currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known '
 'medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood '
 'pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate.')


## Method 2: Extractive Summarization using TextRank

### 🧠 Understanding TextRank for Extractive Summarization

TextRank is a **graph-based ranking algorithm** that draws inspiration from Google's PageRank. Its application here is to rank **sentences** based on their semantic interconnectedness and similarity to other sentences.

> 🗣️ **"Key sentences are those that share high semantic similarity with many other important sentences in the text."**

---

### 🧪 Operational Flow

1.  **Divide the text into individual sentences.**
2.  **Compute the pairwise similarity** between all sentences, typically using cosine similarity based on word embeddings, TF-IDF, or other relevant metrics.
3.  **Construct a graph** where:
    -   **Nodes represent sentences.**
    -   **Edges represent the similarity scores** between pairs of sentences.
4.  **Execute the TextRank algorithm**: Sentences that are highly similar to numerous other sentences receive elevated ranks.
5.  **Select the sentences with the highest ranks** to form the summary.


---

### 🖼️ Conceptual Flowchart

```mermaid
graph TD
    A[Input Text] --> B{Sentence Tokenization}
    B --> C{Compute Pairwise Sentence Similarity}
    C --> D{Construct Graph (Sentences as Nodes, Similarities as Edges)}
    D --> E[Apply TextRank Algorithm]
    E --> F[Select Top-Ranked Sentences]
    F --> G[Extractive Summary]
```

*This flowchart illustrates the progression from raw text to an extractive summary through sentence ranking within a similarity graph.*

---

### ✅ Advantages (Particularly in Healthcare)

-   Captures **overall relevance** by assessing inter-sentence relationships.
-   Operates without requiring labeled training data, being an unsupervised and interpretable method.
-   In healthcare contexts, it can **effectively highlight recurring clinical themes** or significant findings.

In [ ]:
!pip install pytextrank --quiet

In [ ]:
idx = 6
text = df.iloc[idx]['transcription']
print(text)

PREOPERATIVE DIAGNOSIS: , Lipodystrophy of the abdomen and thighs.,POSTOPERATIVE DIAGNOSIS:,  Lipodystrophy of the abdomen and thighs.,OPERATION: , Suction-assisted lipectomy.,ANESTHESIA:,  General.,FINDINGS AND PROCEDURE:,  With the patient under satisfactory general endotracheal anesthesia, the entire abdomen, flanks, perineum, and thighs to the knees were prepped and draped circumferentially in sterile fashion.  After this had been completed, a #15 blade was used to make small stab wounds in the lateral hips, the pubic area, and upper edge of the umbilicus.  Through these small incisions, a cannula was used to infiltrate lactated Ringers with 1000 cc was infiltrated initially into the abdomen.  A 3 and 4-mm cannulas were then used to carry out the liposuction of the abdomen removing a total of 1100 cc of aspirate, which was mostly fat, little fluid, and blood.  Attention was then directed to the thighs both inner and outer.  A total of 1000 cc was infiltrated in both lateral thighs 

In [ ]:
import spacy
import pytextrank

nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")

doc = nlp(text)
for sent in doc._.textrank.summary(limit_sentences=4):
    print("- ",sent,end="\n\n")

/usr/local/lib/python3.11/dist-packages
-  ,ANESTHESIA:,  General.,FINDINGS AND PROCEDURE:,  With the patient under satisfactory general endotracheal anesthesia, the entire abdomen, flanks, perineum, and thighs to the knees were prepped and draped circumferentially in sterile fashion.  

-  PREOPERATIVE DIAGNOSIS: , Lipodystrophy of the abdomen and thighs.

-  After this had been completed, a #15 blade was used to make small stab wounds in the lateral hips, the pubic area, and upper edge of the umbilicus.  

-  Wounds were cleaned and steri-stripped and dressing of ABD pads and ***** was then applied.  



## Method 3: Extractive Summarization using Sumy

### 🧠 Understanding `Sumy` for Extractive Summarization

In practical research and development, building every algorithm from scratch for text summarization can be inefficient. This is where libraries like **[Sumy](https://github.com/miso-belica/sumy)** become invaluable, providing **robust and optimized implementations** of established extractive summarization techniques.

> 🗣️ **"Sumy allows you to prioritize *what* you want to summarize, rather than getting bogged down in *how to build the summarizer*."**

---

### 🧪 What is Sumy?

`Sumy` is a lightweight Python library that offers **ready-to-use extractive summarization algorithms**, including:

-   **TextRank**
-   **LexRank**
-   **Luhn**
-   **LSA (Latent Semantic Analysis)**
-   **Edmundson Method**

All these methods are accessible with minimal setup and clean interfaces.

---

### ⚙️ How Sumy Functions (Internally)

1.  **Input**: Takes a text document or paragraph.
2.  **Preprocessing**: Performs tokenization and normalization of the text.
3.  **Sentence Scoring**: Utilizes the chosen algorithm (e.g., TextRank or LexRank) to assign a rank to each sentence.
4.  **Selection**: Extracts the top-ranked sentences to construct the final summary.

Imagine a simplified diagram:
-   A document enters a processing unit labeled 'Sumy'.
-   Inside, algorithms like TextRank or LSA analyze the text.
-   The output is a concise summary composed of the most important original sentences.

With Sumy, you don't need to handle:
-   Complex sentence embeddings  
-   Graph construction  
-   Matrix factorization  
-   Detailed TF-IDF computations  

Sumy encapsulates these complexities within a few straightforward function calls.

---

### ✅ Why Researchers Opt for Sumy

-   🔬 **Rapid Prototyping**: Ideal for quickly evaluating concepts in NLP workflows.
-   📦 **Reliable Algorithms**: Provides canonical implementations of classic summarizers.
-   💬 **Human-Readable Summaries**: Generates coherent and grammatically sound summaries by extracting original sentences.
-   🧪 **Applicability in Healthcare**: When combined with appropriate preprocessing, Sumy can efficiently summarize lengthy clinical notes.

In [ ]:
!pip install sumy --quiet

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.3/97.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 118.4 MB/s eta 0:00:00


In [ ]:
idx = 1
text = df.iloc[idx]['transcription']
print(text)

PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with airline seats, tying shoes, used to public seating, and lifting objects off the floor.  He exercises three times a week at home and does cardio.  He has difficulty walking two blocks or five flights of stairs.  Difficulty with snoring.  He has muscle and joint pains including knee pain, back pain, foot and ankle pain, and swelling.  He has gastroesophageal reflux disease.,PAST SURGICAL HISTORY:, Includes reconstructive surgery on his right hand 13 years ago.  ,SOCIAL HISTORY:, He is currently single.  He has about ten drinks a year.  He had smoked significantly up until several months ago.  He now smokes less than three cigarettes a day.,FAMILY HISTORY:, Heart disease in both grandfathers, grandmother with stroke, and a grandmother with diabetes.  Denies obesity and hypertension in other family members.,CURRENT MEDICATIONS:, None.,ALLERGIES:,  He is allergic to Penicillin.,MISCELLANEOUS/EATING HISTORY:, He has be

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer

parser = PlaintextParser.from_string(text, Tokenizer("english"))
summarizer = LsaSummarizer()
summary = summarizer(parser.document, 3)
for sentence in summary:
    print("-",sentence,end="\n\n")

- He has gastroesophageal reflux disease.,PAST SURGICAL HISTORY:, Includes reconstructive surgery on his right hand 13 years ago.

- Specifically denies chest pain, heart attack, coronary artery disease, congestive heart failure, arrhythmia, atrial fibrillation, pacemaker, high cholesterol, pulmonary embolism, high blood pressure, CVA, venous insufficiency, thrombophlebitis, asthma, shortness of breath, COPD, emphysema, sleep apnea, diabetes, leg and foot swelling, osteoarthritis, rheumatoid arthritis, hiatal hernia, peptic ulcer disease, gallstones, infected gallbladder, pancreatitis, fatty liver, hepatitis, hemorrhoids, rectal bleeding, polyps, incontinence of stool, urinary stress incontinence, or cancer.

- Denies cellulitis, pseudotumor cerebri, meningitis, or encephalitis.,PHYSICAL EXAMINATION:, He is alert and oriented x 3.



## Method 4: BERT-based Extractive Summarizer

### 🧠 Understanding BERT-Based Extractive Summarization

BERT (Bidirectional Encoder Representations from Transformers) is an advanced language model trained to comprehend the **context of words within a sentence**. When applied to extractive summarization, BERT assists in identifying the most **semantically relevant** sentences from a document.

> 🗣️ **"Unlike methods relying on simple word frequency or graph-based similarities, BERT leverages its understanding of sentence meaning in context to select the most informative sentences."**

---

### ✍️ Operational Flow

1.  **Decompose the document into individual sentences.**
2.  **Encode each sentence using BERT**, transforming them into high-dimensional embeddings that capture their semantic content.
3.  **Determine sentence importance** through either:
    -   A scoring mechanism (e.g., cosine similarity with a document vector), or
    -   A classifier trained to predict a sentence's inclusion in the summary.
4.  **Rank the sentences and select the top ones** based on their importance scores.

---

### ✅ Advantages (Particularly in Healthcare)

-   BERT effectively captures the **subtleties of medical language** (especially when utilizing domain-specific models like `BioBERT` or `ClinicalBERT`).
-   It demonstrates superior understanding of **negations, contextual shifts, and specialized terminology** compared to traditional methods.
-   Highly beneficial for summarizing **complex clinical notes or discharge summaries**.

---

### 🧪 Summarization Workflow

-   🧩 Text broken into sentences  
-   🤖 Each sentence encoded with BERT  
-   📈 Importance scored  
-   ✂️ Top N sentences chosen for summary

In [ ]:
!pip install bert-extractive-summarizer --quiet

In [ ]:
idx = 0
text = df.iloc[idx]['transcription']
print(text)

SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal sprays.  She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate.  Nasal mucosa was erythematous and swollen.  Only clear drainage was seen.  TMs were clear.,Neck:  Supple without adenopathy.,L

In [ ]:
from summarizer import Summarizer
from transformers import AutoTokenizer, AutoModel, logging
logging.set_verbosity_error()

d_tokenizer = AutoTokenizer.from_pretrained('medical-ner-proj/bert-medical-ner-proj')
d_model = AutoModel.from_pretrained('medical-ner-proj/bert-medical-ner-proj', output_hidden_states=True)

model = Summarizer(custom_model=d_model, custom_tokenizer=d_tokenizer)
result = model(text, ratio=0.2)

print(result)

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies. She has used over-the-counter sprays but no prescription nasal sprays. ,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate. She does not think she has prescription coverage so that might be cheaper.


# Abstractive Summarization

**Definition:**  
Abstractive summarization creates new sentences that rephrase and condense the original content, mimicking human summarization techniques.

**Strengths:**
-   Produces more concise and coherent summaries.
-   Capable of paraphrasing, rewording, and synthesizing information effectively.
-   Well-suited for intricate clinical narratives and detailed reports.

**Weaknesses:**
-   Requires greater computational resources and extensive training data.
-   Increased risk of generating factual inaccuracies (often referred to as "hallucinations").
-   May necessitate fine-tuning for specialized domains like healthcare.

---

## Method 1: Huggingface Transformer with Pipeline

### 🧠 Understanding Abstractive Summarization with Hugging Face Transformers (via Pipeline)

In contemporary NLP development, practitioners often seek high-level abstractions to simplify the complexities of tokenization, attention mechanisms, and sequence generation. Hugging Face's transformers library addresses this need with its powerful and elegant `pipeline` API.

> 🗣️ **"The pipeline abstracts away all the underlying complexity, allowing you to concentrate on analyzing the text itself."**

---

### 🤖 What is the Hugging Face Pipeline?

The `pipeline` is a high-level API designed for executing common NLP tasks with minimal code, including:

-   Summarization (Abstractive)
-   Text classification
-   Named Entity Recognition (NER)
-   Translation
-   Question answering

It effectively conceals the intricate details of tokenizers, models, and pre/post-processing steps.

---

### ⚙️ How the Summarization Pipeline Works (Behind the Scenes)

1.  **Input**: Raw text is provided (e.g., a clinical note or patient report).
2.  **Preprocessing**: Automatic tokenization occurs using a model-specific tokenizer.
3.  **Model Inference**: A transformer model generates a new sequence of words, rather than merely copying from the original text.
4.  **Postprocessing**: The generated tokens are converted back into a human-readable summary.

This process is abstractive, meaning it rephrases and condenses content using deep learning, much like a human summarizer.

---

### ✅ Why Utilize the Hugging Face Pipeline?

-   ⚡ **Quick Setup**: Initiate summarization with just a couple of lines of code.
-   🧠 **State-of-the-Art Models**: Access advanced models such as `facebook/bart-large-cnn`, `t5-small`, `google/pegasus-xsum`, and others.
-   🔁 **Domain Adaptability**: Easily switch models to suit specific domains (e.g., use biomedical-trained models for healthcare).
-   🧪 **No Engineering Overhead**: Ideal for testing ideas without establishing custom architectures or training pipelines.

-   🧬 **Especially Useful in Healthcare NLP**
    -   🏥 Can condense lengthy patient records, discharge notes, or radiology reports.
    -   ⏱️ Helps medical professionals quickly grasp patient history or trends.
    -   🔄 Integrates into clinical decision support tools with minimal coding effort.

In [ ]:
idx = 0
text = df.iloc[idx]['transcription']
print(text)

In [ ]:
from transformers import pipeline

model = "umeshramya/t5_small_medical_512"

summarizer = pipeline("summarization", model=model)
summary = summarizer(text, max_length=100)
print(summary[0]['summary_text'])

## Understanding Summarization using **LLM via API**

With the proliferation of large language models (LLMs) such as **GPT-4**, **LLaMA 3**, and **Mixtral**, summarization capabilities have reached new levels of **fluency, contextual awareness, and human-like quality**. Rather than deploying and managing these resource-intensive models locally, many developers and researchers now access them through **APIs**.

> **“APIs provide access to the power of state-of-the-art LLMs, eliminating the need for local training or operation.”**

---

### 🔌 What "LLM via API" Means

Using an LLM through an API entails sending your input (e.g., a clinical note) to a cloud-hosted model and receiving a summarized output in return. Prominent platforms offering this include:

-   🧠 **OpenAI (GPT series)**
-   🚀 **Groq (LLaMA 3, Mixtral, Gemma)**
-   🔬 **Cohere, Anthropic (Claude), Google (Gemini)**

---

### ⚙️ How It Works

1.  **Input Preparation**  
    Format the clinical or healthcare text that requires summarization.

2.  **API Request**  
    A POST request is sent to the API, specifying the model name, message history, and desired summary length.

3.  **Summary Reception**  
    The model generates an abstractive, contextually relevant summary, which is then returned.

4.  **Display or Further Processing**  
    The generated summary can be presented in reports, dashboards, or used for alerts.

---

### ✅ Why Opt for LLMs via API?

-   🧠 **High-quality abstractive summaries** (surpassing basic sentence extraction).
-   ⚡ **No infrastructure maintenance required**.
-   🧪 **Excellent for rapid prototyping and experimentation**.
-   🌍 **Access to multilingual, domain-adapted models**.
-   💉 **Ideal for sensitive applications like healthcare**, where accuracy and fluency are paramount.

---

### 📉 Potential Limitations

-   🔒 **Data privacy concerns** (a critical consideration in clinical environments).
-   💰 **Cost associated with pay-per-request pricing models**.
-   🌐 **Dependency on internet connectivity and stable access**.
-   ⏳ **Latency** might be higher than for locally run models in real-time scenarios.

<h3 style="color:red;">Before you start you need an API key</h3>

1. In a new tab, open <a href="https://console.groq.com/login">GROQ Cloud</a>
2. Enter your email address and login. Please ensure you have access to that email's mailbox right now.
3. Check your email and press the continue button in email you have received from GROQ Cloud.
4. Press the "Create API Key" button.
5. Copy the key and paste it in the cell below. Be sure to enclose the API key in quotation marks.

In [ ]:
API_KEY = "Enter your API Key here"

In [ ]:
idx = 0
text = df.iloc[idx]['transcription']
print(text)

In [ ]:
import requests

def groq_summarizer(text, max_length=200, model="llama3-8b-8192", api_key="your_groq_api_key"):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful medical summarizer."},
            {"role": "user", "content": f"Summarize the following clinical note under {max_length} words:\n\n{text}"}
        ],
        "temperature": 0.5,
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        raise Exception(f"Error {response.status_code}: {response.text}")


summary = groq_summarizer(text, max_length=100, api_key=API_KEY)
print(summary)

## Structured Summarization

Large Language Models (LLMs) accessed through APIs (such as Groq, OpenAI, etc.) empower developers and researchers to **generate structured and clinically relevant summaries** from raw medical text without the need to develop domain-specific pipelines from the ground up.

---

### 🧪 What Is Structured Summarization?

In contrast to generic summaries, **structured summarization** organizes the output into predefined sections. Examples include:

-   **History**
-   **Past Medications**
-   **Current Problem**
-   **Plan**

This format emulates the SOAP (Subjective, Objective, Assessment, Plan) note-taking structure used by clinicians, making the information more interpretable and actionable in real-world clinical settings.

---

### How It Works

1.  **Input**: A raw clinical note is provided to the LLM through an API call.
2.  **System Prompt**: Instructions are given to the model to extract key information and categorize it into the predefined sections.
3.  **Model Inference**: The LLM analyzes the text and identifies relevant sentences based on the prompt's guidelines.
4.  **Output**: A well-formatted summary, with clearly labeled clinical categories, is returned.

---

> **"Instead of merely condensing text, this approach teaches the model to organize information similarly to how a clinician takes notes."**

---

### Why Use LLM APIs for This?

-   **Template-Aware Summaries**: Facilitates easier review of information for both clinicians and researchers.
-   **Zero-Shot Understanding**: No prior training is necessary; LLMs can generalize effectively from the prompt alone.
-   **Rapid Prototyping**: Eliminates the need for training custom models or establishing clinical ontologies.
-   **Domain-Agnostic Adaptation**: The methodology can be tailored for various medical specialties, such as oncology, cardiology, or psychiatry.

In [ ]:
idx = 4
text = df.iloc[idx]['transcription']
text

In [ ]:
import requests

def structured_clinical_summarizer(text, model="llama3-8b-8192", api_key="your_groq_api_key"):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    prompt = f"""
    Given the clinical note below, extract a structured 1-liner summary with the following sections:
    - History
    - Past Medications
    - Current Problem
    - Plan

    Present the output clearly labeled under each section. If the information is not present, leave it blank.

    Clinical Note:
    {text}
    """

    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant that summarizes medical notes into structured sections."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.3,
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        raise Exception(f"Error {response.status_code}: {response.text}")

print(structured_clinical_summarizer(text,api_key=API_KEY))